# Etapa 2 — Tratamento de dados (fundação)

**Projeto:** canavis — benchmarking regional da cana-de-açúcar na Paraíba  
**Usuário final:** Ricardo, gerente agrícola de usina

Este notebook de diagnóstico resolve as pendências de dados levantadas no feedback da Etapa 1, **antes** de construir qualquer visual em Power BI. Responde a três frentes:

1. Confirmar o último ano da PAM e separar **período de referência** de **data de publicação** (defasagem).
2. Classificar cada ausência municipal em **sem produção**, **suprimido** ou **outra condição**, por regra explícita derivada do token do IBGE — nunca do porte do município.
3. Revisar o recorte territorial: relação entre **microrregião** e as regiões geográficas **imediata/intermediária** (DRB 2017) e a regra de agrupamento estável ao longo da série.

A lógica mora no pacote (`canavis.pipeline.transform`, `canavis.domain.territorio`); o que se demonstra aqui é reproduzível e testado. Decisões registradas no ADR `docs/decisoes/0008-tratamento-etapa2.md`.

In [1]:
import pandas as pd

from canavis import storage
from canavis.config import load_settings
from canavis.pipeline.transform import classificar_ausencia
from canavis.domain import territorio

pd.set_option("display.max_columns", None)

# Lê o dado cru (camada bronze): os tokens de ausência ainda estão presentes.
raw = load_settings().paths.raw
pam = storage.read(raw, "ibge_pam")
pam.shape

(28665, 6)

## 1. Último ano da PAM: período de referência × data de publicação

O último ano é confirmado **programaticamente** por `max(periodo)`. Esse é o **período de referência** (ano a que o dado se refere). A PAM é divulgada com cerca de **um ano de defasagem**: o ano-referência 2023 foi publicado em 2024. Portanto o ano mais recente da série não é o ano corrente — distinção que todo visual deve preservar.

In [2]:
periodos = pd.to_numeric(pam["periodo"], errors="coerce")
ultimo_ano = int(periodos.max())
primeiro_ano = int(periodos.min())
data_publicacao_aprox = ultimo_ano + 1  # defasagem de ~1 ano da PAM

print(f"Série PAM: {primeiro_ano}–{ultimo_ano}")
print(f"Período de referência (ano do dado): {ultimo_ano}")
print(f"Data de publicação aproximada (defasagem ~1 ano): {data_publicacao_aprox}")

Série PAM: 2003–2023
Período de referência (ano do dado): 2023
Data de publicação aproximada (defasagem ~1 ano): 2024


## 2. Classificação de ausência: sem produção × suprimido × outra condição

A Etapa 1 colapsava toda ausência num único balde descartado. A regra nova deriva a situação **do token publicado pelo IBGE (SIDRA)**, nunca do porte do município:

| Token | Significado IBGE | Rótulo |
|-------|------------------|--------|
| numérico | dado informado | `presente` |
| `-` | zero absoluto, fenômeno não existe | `sem_producao` |
| `...` | dado não disponível (inclui sigilo) | `suprimido` |
| `X` | omitido para não identificar o informante | `suprimido` |
| `..` | não se aplica | `outra_condicao` |
| vazio/None/NaN | ausência sem token semântico | `outra_condicao` |

Primeiro, a distribuição dos tokens crus por nível (`extracao`) e por `variavel`.

In [3]:
# Tokens não-numéricos que aparecem no valor cru.
valor_str = pam["valor"].astype("string").str.strip()
tokens = ["-", "..", "...", "X"]
crus = valor_str[valor_str.isin(tokens)]
print("Contagem de tokens no dado cru:")
print(crus.value_counts().to_dict())
print("\nTokens por nível (extracao):")
display(
    pam.assign(token=valor_str.where(valor_str.isin(tokens)))
    .dropna(subset=["token"])
    .pivot_table(index="extracao", columns="token", values="valor", aggfunc="count", fill_value=0)
)

Contagem de tokens no dado cru:
{'-': 15176, '...': 130}

Tokens por nível (extracao):


token,-,...
extracao,,
serie_microrregioes_pb,880,0
serie_municipios_pb,14296,130


In [4]:
# Aplica a regra de classificação a cada valor cru.
pam = pam.assign(situacao_valor=pam["valor"].map(classificar_ausencia))

print("Classificação por nível territorial:")
display(
    pam.pivot_table(
        index="extracao", columns="situacao_valor", values="valor",
        aggfunc="count", fill_value=0,
    )
)

Classificação por nível territorial:


situacao_valor,presente,sem_producao,suprimido
extracao,,,
serie_microrregioes_pb,1535,880,0
serie_municipios_pb,8989,14296,130
serie_ufs_brasil,2835,0,0


A ausência de ~62% no nível municipal da Etapa 1 deixa de ser um bloco único. Abaixo, a composição da ausência no nível **município**: separar `sem_producao` (zero real, plotável como zero) de `suprimido` (dado sonegado, exibido como "sem informação") muda o que o visual pode afirmar.

In [5]:
muni = pam[pam["extracao"] == "serie_municipios_pb"]
composicao = muni["situacao_valor"].value_counts()
proporcao = (composicao / len(muni) * 100).round(1)
resumo_muni = pd.DataFrame({"linhas": composicao, "% do nível": proporcao})
print(f"Nível município — {len(muni)} linhas")
display(resumo_muni)
ausencia_total = proporcao.drop("presente", errors="ignore").sum()
print(f"Ausência total (não-presente) no município: {ausencia_total:.1f}%")

Nível município — 23415 linhas


,linhas,% do nível
situacao_valor,,
sem_producao,14296,61.1
presente,8989,38.4
suprimido,130,0.6


Ausência total (não-presente) no município: 61.7%


## 3. Recorte territorial: microrregião × regiões imediata/intermediária (DRB 2017)

Em 2017 o IBGE substituiu meso/microrregiões pela **Divisão Regional do Brasil (DRB 2017)**, com regiões geográficas **imediatas** e **intermediárias**. A série da PAM vai de **2003 a 2023** e atravessa essa mudança.

**Chave estável de agrupamento:** a **microrregião**. É a única unidade sub-UF presente em toda a série com completude alta o bastante (ADR 0004). A DRB 2017 só existe a partir de 2017, então **não** substitui a chave: o de-para abaixo serve para *comparar* a leitura antiga com as divisões novas sem reprocessar a série.

As cinco microrregiões canavieiras (98% da produção da PB, ADR 0007) e seu de-para para as regiões geográficas. ⚠️ O mapeamento é ao **nível de nome** e está **pendente de confirmação na fonte oficial do IBGE** — nenhum código oficial foi inventado.

In [6]:
print("Confirmação em fonte oficial pendente:", territorio.CONFIRMACAO_FONTE_OFICIAL_PENDENTE)

de_para = pd.DataFrame(
    [
        {
            "microrregiao": micro,
            "imediata": regiao["imediata"],
            "intermediaria": regiao["intermediaria"],
            "cod_imediata": regiao["cod_imediata"],
            "cod_intermediaria": regiao["cod_intermediaria"],
        }
        for micro, regiao in territorio.MICRORREGIOES_CANAVIEIRAS.items()
    ]
)
display(de_para)
print(
    "\nCódigos oficiais ainda não verificados (None) — células que precisam\n"
    "de confirmação na Divisão Regional do Brasil 2017 do IBGE."
)

Confirmação em fonte oficial pendente: True


,microrregiao,imediata,intermediaria,cod_imediata,cod_intermediaria
0,Litoral Norte - PB,Mamanguape,João Pessoa,None,None
1,Litoral Sul - PB,João Pessoa,João Pessoa,None,None
2,Sapé - PB,João Pessoa,João Pessoa,None,None
3,João Pessoa - PB,João Pessoa,João Pessoa,None,None
4,Brejo Paraibano - PB,Guarabira,João Pessoa,None,None



Códigos oficiais ainda não verificados (None) — células que precisam
de confirmação na Divisão Regional do Brasil 2017 do IBGE.


### Síntese

- **Último ano:** confirmado programaticamente; período de referência (2023) ≠ data de publicação (~2024, defasagem de ~1 ano).
- **Ausência:** classificada por token IBGE em `sem_producao` / `suprimido` / `outra_condicao`, nunca pelo porte do município — `-` vira zero real, `...`/`X` ficam como "sem informação".
- **Território:** microrregião é a chave estável 2003–2023; de-para para imediata/intermediária documentado ao nível de nome, com confirmação oficial pendente sinalizada.

Regras registradas no ADR `docs/decisoes/0008-tratamento-etapa2.md`.